# 02 · Observational causal inference, uplift modelling, and the targeting policy

Part 1 established the experimental ground truth, in particular, the
Women's e-mail raises the two-week visit rate by **+4.52pp (95% CI 3.90 to
5.16)**. This notebook uses that truth three ways:

* **Part 2**: sabotage the randomisation on purpose, then recover the truth
  with observational methods (the methodological showpiece).
* **Part 3**: estimate *heterogeneous* effects with from-scratch uplift
  meta-learners and evaluate them with Qini curves.
* **Part 4**: convert scores into an e-mail targeting policy with a profit
  simulation.

In [ ]:
import sys

sys.path.append("..")

import json

import numpy as np
import pandas as pd

from src import config
from src.data_ingestion import load_clean

df = load_clean()
benchmark = json.loads((config.REPORTS / "rct_summary.json").read_text())["benchmark"]
benchmark

## Part 2, manufacture confounding, then defeat it

Real marketing data is rarely randomised: campaigns target the "best"
customers, so treatment correlates with everything. We recreate that world
by **biased subsampling** of the Women's-e-mail + control cell. With
standardised $z_h$ = log-history and $z_r$ = recency, each unit's *true*
selection propensity is

$$e^*(x) = \mathrm{clip}\big(\sigma(-0.20 + 0.90\,z_h - 0.70\,z_r +
0.50\,\text{multichannel}),\ 0.03,\ 0.97\big)$$

Treated units are kept with probability $e^*(x)$, controls with
$1 - e^*(x)$: engaged customers are now over-represented among the treated.
Crucially, selection depends only on *observed* covariates, so ignorability
holds by construction and the RCT ATE is still the estimand's true value.

In [ ]:
from src.observational import (
    aipw,
    fit_propensity,
    ipw,
    make_confounded,
    naive_diff,
    nn_matching,
    regression_adjustment,
)

obs = make_confounded(df)
naive = naive_diff(obs)
print(f"confounded sample n={len(obs):,}, treated={int(obs['W'].sum()):,}")
print(f"naive DIM: {naive['estimate']:+.4f}  vs RCT truth {benchmark['ate']:+.4f}"
      f"  (bias {naive['estimate'] - benchmark['ate']:+.4f})")

The naive difference in means overstates the true effect by ~3.1pp, a
**+68% exaggeration** that would badly oversell the campaign. Four
adjustment strategies, all assuming selection-on-observables:

1. **Regression adjustment**: OLS of $Y$ on $(W, X)$ with HC1 robust SEs.
2. **IPW**: logistic propensity $\hat{e}(x)$, trimmed to $[0.02, 0.98]$,
   with *stabilised* weights $w_i = \tfrac{p}{\hat e(x_i)}$ (treated) and
   $\tfrac{1-p}{1-\hat e(x_i)}$ (controls), Hájek-normalised; bootstrap CI
   refits the propensity model inside every replicate.
3. **1:1 nearest-neighbour matching** on the logit propensity (caliper 0.2
   SD, with replacement), estimates the **ATT**, not the ATE.
4. **AIPW / doubly robust**, cross-fitted:

$$\hat\psi_i = \hat m_1(X_i) - \hat m_0(X_i)
 + \frac{W_i\,(Y_i - \hat m_1(X_i))}{\hat e(X_i)}
 - \frac{(1-W_i)\,(Y_i - \hat m_0(X_i))}{1 - \hat e(X_i)},
 \qquad \widehat{\mathrm{ATE}} = \tfrac1n\sum_i \hat\psi_i.$$

AIPW is consistent if *either* the propensity *or* the outcome model is
right, and its influence-function form gives an analytic SE.

In [ ]:
results = {"naive": naive,
           "regression adjustment": regression_adjustment(obs)}
ipw_res, e_hat = ipw(obs)
results["IPW (stabilised, trimmed)"] = ipw_res
match_res, matched = nn_matching(obs, e_hat)
results["1:1 NN matching (ATT)"] = match_res
results["AIPW (cross-fit)"] = aipw(obs)

comp = pd.DataFrame(
    [{"method": k, "estimate": v["estimate"], "ci_lo": v["ci_lo"],
      "ci_hi": v["ci_hi"],
      "abs_error_vs_RCT": abs(v["estimate"] - benchmark["ate"])}
     for k, v in results.items()]
)
comp.round(4)

On the real data every adjustment method lands within ~0.1pp of the RCT
truth, except matching, which sits ~0.9pp above it. That is not a failure:
matching targets the **ATT**, and the treated (post-selection) are engaged
customers whose uplift *is* genuinely larger (Part 1 heterogeneity). The
estimand, not the estimator, differs.

Diagnostics worth showing in any observational study: propensity overlap,
the love plot (|SMD| before/after adjustment), and the stabilised-weight
distribution, see `reports/figures/fig_obs_*.png`, regenerated by
`python -m src.observational`.

## Part 3, uplift meta-learners (from scratch)

Estimand: the CATE $\tau(x) = E[Y(1) - Y(0) \mid X = x]$ for the Women's
e-mail on `visit`. Four learners, all thin classes over LightGBM
(`src/uplift/meta_learners.py`):

* **S-learner**: one model $f(X, W)$; $\hat\tau(x) = f(x,1) - f(x,0)$.
* **T-learner**: $f_1$ on treated, $f_0$ on controls;
  $\hat\tau = f_1 - f_0$.
* **X-learner**: T-learner, then regress imputed effects
  $D^1 = Y - f_0(X)$, $D^0 = f_1(X) - Y$ on $X$; combine
  $\hat\tau = e\,\hat\tau_0 + (1-e)\,\hat\tau_1$.
* **Class transformation**: $Z = WY + (1-W)(1-Y)$;
  $\hat\tau = 2\,\hat P(Z=1\mid x) - 1$ (valid because assignment is
  ~50/50 in this cell).

Evaluation is ranking-based. Sorting customers by predicted uplift, the
**Qini curve** tracks estimated incremental visits among the top-$n$:

$$Q(n) = Y_t(n) - Y_c(n)\,\frac{n_t(n)}{n_c(n)},$$

and the **Qini coefficient** is the area between $Q$ and the
random-targeting diagonal. Implemented from scratch in
`src/uplift/qini.py`, with a seeded tie-break so results are reproducible.

In [ ]:
from sklearn.model_selection import train_test_split

from src.uplift.meta_learners import (
    ClassTransformation, SLearner, TLearner, XLearner,
)
from src.uplift.qini import qini_coefficient, qini_curve, uplift_at_k
from src.uplift.run_uplift import load_cell

sub, X, y, w = load_cell()
strata = 2 * w + y
idx_tr, idx_te = train_test_split(
    np.arange(len(y)), test_size=config.TEST_SIZE, stratify=strata,
    random_state=config.SEED,
)

learners = {"S-learner": SLearner(), "T-learner": TLearner(),
            "X-learner": XLearner(), "class-transformation": ClassTransformation()}
scores = {}
for name, model in learners.items():
    model.fit(X[idx_tr], w[idx_tr], y[idx_tr])
    scores[name] = model.predict_uplift(X[idx_te])
    print(f"{name:22s} Qini={qini_coefficient(y[idx_te], w[idx_te], scores[name]):7.2f}"
          f"  uplift@10%={uplift_at_k(y[idx_te], w[idx_te], scores[name], 0.10):.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
for name, s in scores.items():
    frac, q = qini_curve(y[idx_te], w[idx_te], s)
    step = max(1, len(frac) // 400)
    ax.plot(frac[::step], q[::step], label=name)
ax.plot([0, 1], [0, q[-1]], ls="--", color="gray", label="random")
ax.set_xlabel("fraction targeted")
ax.set_ylabel("incremental visits (Qini)")
ax.legend()
plt.show()

On the held-out test split the **S-learner** ranks best (Qini ≈ 55; top
decile uplift ≈ +8.5pp vs the +4.5pp average) and the full pipeline's
1,000-resample bootstrap puts its Qini advantage over the runner-up
X-learner at **+23.6 [9.4, 37.4]**: a real, not cosmetic, difference. With
modest heterogeneity, the S-learner's shrinkage towards a common effect is
an advantage; the T-learner's independent arms overfit arm-specific noise,
and the class-transformation estimator is noisiest, as expected from its
variance-inflating construction.

The pipeline (`python -m src.uplift.run_uplift`) additionally computes
5-fold out-of-fold scores within the training split (an overfitting check)
and out-of-fold scores over the *full* cell, the input Part 4 uses, so no
customer is ever scored by a model that saw their outcome.

## Part 4, the targeting policy: modelling → money

Profit model with margin $M$ per incremental conversion and cost $c$ per
e-mail, targeting the top-$k\%$ ($m_k$ customers):

$$\pi(k) = m_k\,\big(\widehat{\Delta\mathrm{conv}}_k \cdot M - c\big)$$

Because the underlying data is randomised,
$\widehat{\Delta\mathrm{conv}}_k$ is estimated *empirically*, treated
minus control conversion rates **within the targeted group**: an offline
policy-value estimate that uses the model only for ranking.

In [ ]:
from src.policy import compute_profit_curve

scores_df = pd.read_csv(config.SCORES_FILE)
curve = compute_profit_curve(scores_df,
                             margin=config.MARGIN_PER_CONVERSION,
                             cost=config.COST_PER_EMAIL)
k_star = int(curve.loc[curve["profit_topk"].idxmax(), "k_pct"])

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(curve["k_pct"], curve["profit_topk"], label="top-k% by uplift")
ax.plot(curve["k_pct"], curve["profit_random"], ls="--", color="gray",
        label="random k%")
ax.axhline(0, color="black", lw=1)
ax.scatter([k_star], [curve["profit_topk"].max()], color="red", zorder=5,
           label=f"k* = {k_star}%")
ax.set_xlabel("share of customers e-mailed (%)")
ax.set_ylabel("expected profit ($)")
ax.legend()
plt.show()

curve.loc[curve["k_pct"].isin([10, 20, k_star, 50, 100]),
          ["k_pct", "n_targeted", "inc_conv_rate", "profit_topk", "profit_random"]]

At the default economics ($25 margin, $0.10 per e-mail) the punchline on
real data is stark: **blanket e-mailing loses ≈ $950**, random targeting of
a third of the base still loses money, while e-mailing the **top 35% by
predicted uplift makes ≈ +$1,065**, the same campaign turned from
value-destroying to value-creating purely by *choosing who*.

Caveats that keep this honest: conversions are rare, so the profit CI is
wide (bootstrap ≈ [−$36, +$2,238]); the margin/cost parameters are
assumptions (sweep them in the Streamlit app); the ranking optimises visit
uplift while profit prices conversions; and a two-week window cannot speak
to long-term effects like e-mail fatigue.